In [1]:
%load_ext autoreload
%autoreload 2

In [4]:
import os
from moc.configs.config import get_config
from moc.utils.run_config import RunConfig
# from moc.models.mqf2.lightning_module import MQF2LightningModule
from moc.models.mixture.mixture_model2 import MixtureLightningModule
from moc.models.gaussian.gaussian import GaussianLightningModule
from moc.models.trainers.lightning_trainer import get_lightning_trainer
from moc.datamodules.real_datamodule import RealDataModule
import numpy as np
import matplotlib.pyplot as plt
from moc.metrics.distribution_metrics import pce, multivariate_energy_score, mse
import torch
from pathlib import Path


In [5]:
import collections
torch.set_printoptions(precision=3, sci_mode=False, threshold=float('inf'), edgeitems=40, linewidth=200)
np.set_printoptions(suppress=True, formatter={'float_kind':'{:.3f}'.format})

In [ ]:

config = get_config()
config.device = 'cuda'
 
datasets = [
    ('mulan', 'scm20d'),
    ('mulan', 'wq'),
    ('cevid', 'air'),
    ('cevid', 'births2'),
    ('camehl', 'households')  # correction de la parenthèse
]
 
preranks_train = ['marginal', 'mean', 'variance', 'dependency', 'pca' ,'density', 'cdf']
prerank_eval = ['marginal', 'mean', 'variance', 'dependency', 'pca', 'density', 'cdf']
seeds = [0, 4, 12, 42, 866]
 
# Pour stocker tous les résultats
global_pce = {}
 
for data_group, data_name in datasets:
    print(f"Running on dataset: {data_group}/{data_name}")
 
    for prerank_train in preranks_train:
        print(f"  Train prerank: {prerank_train}")
        pce_by_prerank = collections.defaultdict(list)
 
        for seed in seeds:
            rc = RunConfig(config, data_group, data_name, seed=seed)
            datamodule = RealDataModule(rc, num_workers=8, seed=seed)
            test_loader = datamodule.test_dataloader()
 
            # Dossier racine
            # root_dir = Path(f"/ssd/bdml2/naomi/Multivariate-recalibration/Multicalibration/Projected_PIT_calibration/log/checkpointsd/prerank/{data_group}/{data_name}")
            root_dir = Path(f"/mnt/default/elnura_workspace/Multivariate-recalibration/Multicalibration/Projected_PIT_calibration/logs/prerank/{data_group}/{data_name}")
            # Chercher le sous-dossier correspondant à la seed et au prerank_train
            matched_dirs = [
                d for d in root_dir.iterdir()
                if d.is_dir() and f",seed={seed}," in d.name and f"prerank={prerank_train}" in d.name
            ]
 
            if len(matched_dirs) != 1:
                print("Matched dirs:")
                for d in matched_dirs:
                    print("  -", d.name)
                raise ValueError(f"Expected one matching directory for seed={seed} and prerank={prerank_train}, found {len(matched_dirs)}.")
 
            ckpt_dir = matched_dirs[0] / "0" / "checkpoints"
            ckpt_files = list(ckpt_dir.glob("*.ckpt"))
 
            if len(ckpt_files) != 1:
                raise ValueError(f"Expected one checkpoint in {ckpt_dir}, found {len(ckpt_files)}.")
           
            ckpt_path = str(ckpt_files[0])
            model = MixtureLightningModule.load_from_checkpoint(ckpt_path)
            model.eval().to(config.device)
 
            for prerank in prerank_eval:
                pces = []
                for x, y, idx in test_loader:
                    x = x.to(config.device)
                    y = y.to(config.device)
                    dist = model.predict(x)
                    pce_val, _ = pce(dist, y, n_samples=model.hparams.es_num_samples, prerank=prerank)
                    pces.append(pce_val)
 
                pce_total = torch.stack(pces).mean(dim=0)
                pce_by_prerank[prerank].append(pce_total.mean())
 
        # Agrégation des résultats pour ce dataset + ce prerank_train
        line = " & ".join([f"{round(torch.stack(pce_by_prerank[p]).mean().item(), 3)}" for p in prerank_eval])
        print(f"{data_group}/{data_name} | train={prerank_train} → {line}")
        global_pce[(data_group, data_name, prerank_train)] = line

In [ ]:
config = get_config()
config.device = 'cuda'
data_group, data_name = 'mulan', 'scm1d'
prerank_train = 'cdf'
prerank_eval = ['marginal', 'mean', 'variance', 'dependency', 'pca', 'density', 'cdf']
seeds = [0, 4, 12, 42, 866]
epochs = [75, 35, 80, 99, 14]
lamda = 5.0
pce_by_prerank = collections.defaultdict(list)

for seed, epoch in zip(seeds, epochs):
    rc = RunConfig(config, data_group, data_name, seed=seed)
    datamodule = RealDataModule(rc, num_workers=8, seed=seed)
    test_loader = datamodule.test_dataloader()
    
    # ckpt_path = f"/mnt/default/elnura_workspace/Multivariate-recalibration/Multicalibration/Projected_PIT_calibration/logs/2025-07-19/18-53-46/{data_group}/{data_name}/model=mixture,seed={seed},lambda=10.0,prerank={prerank_train}/0/checkpoints/epoch_{epoch:04d}.ckpt"
    # ckpt_path_marg_prerank = f"/mnt/default/elnura_workspace/Multivariate-recalibration/Multicalibration/Projected_PIT_calibration/logs/2025-07-22/20-28-27/{data_group}/{data_name}/model=mixture,seed={seed},lambda={lamda},prerank={prerank_train}/0/checkpoints/epoch_{epoch:04d}.ckpt"
    ckpt_path_pca_prerank = f"/mnt/default/elnura_workspace/Multivariate-recalibration/Multicalibration/Projected_PIT_calibration/logs/2025-07-27/13-04-01/{data_group}/{data_name}/model=mixture,seed={seed},lambda={lamda},prerank={prerank_train}/0/checkpoints/epoch_{epoch:04d}.ckpt"
    model = MixtureLightningModule.load_from_checkpoint(ckpt_path_pca_prerank)
    model.eval().to(config.device)

    for prerank in prerank_eval:
        pces = []
        for x, y, idx in test_loader:
            x = x.to(config.device)
            y = y.to(config.device)
            dist = model.predict(x)
            pce_val, _ = pce(dist, y, n_samples=model.hparams.es_num_samples, prerank=prerank)
            pces.append(pce_val)

        pce_total = torch.stack(pces).mean(dim=0)
        pce_by_prerank[prerank].append(pce_total.mean())

# Final aggregation
# for prerank in prerank_eval:
#     stacked = torch.stack(pce_by_prerank[prerank])  # shape: (5, output_dim)
#     avg = stacked.mean(dim=0).item()
#     print(f"model trained on {prerank_train}, {prerank} PCE (avg over seeds): {round(avg, 3)}")
line = " & ".join([f"{round(torch.stack(pce_by_prerank[p]).mean().item(), 3)}" for p in prerank_eval])
print(line)

0.036 & 0.043 & 0.077 & 0.105 & 0.046 & 0.101 & 0.042
